In [2]:
# =====================
# 1. CSV load and setup configs
# =====================

import pandas as pd
import numpy as np
import os
import sys
!{sys.executable} -m pip install ipython nbformat
#!{sys.executable} -m pip install --upgrade nbformat

# Configs
RANDOM_STATE = 555
pd.set_option('display.max_columns', None)

# Directories
input_dir = 'csv_input'
output_dir = 'csv_output'
os.makedirs(output_dir, exist_ok=True)

file_path = os.path.join(input_dir, 'WA_Fn-UseC_-Telco-Customer-Churn.csv')

# Read only column names
all_cols = pd.read_csv(file_path, nrows=0).columns.tolist()
print('All columns:')
print(all_cols)

# Configure list of columns to exclude
exclude_cols = ['customerID']

# Define columns to keep
use_cols = [col for col in all_cols if col not in exclude_cols]

# Load dataset without excluded columns
df_raw = pd.read_csv(file_path, usecols=use_cols)

print(f'\nDataset loaded: {file_path} \nwithout these cols: {exclude_cols}')
print('Shape:', df_raw.shape)
display(df_raw.head())

All columns:
['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn']

Dataset loaded: csv_input\WA_Fn-UseC_-Telco-Customer-Churn.csv 
without these cols: ['customerID']
Shape: (7043, 20)



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: C:\Users\User\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,Female,0,Yes,No,1,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,Male,0,No,No,34,Yes,No,DSL,Yes,No,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,Male,0,No,No,2,Yes,No,DSL,Yes,Yes,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,Male,0,No,No,45,No,No phone service,DSL,Yes,No,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,Female,0,No,No,2,Yes,No,Fiber optic,No,No,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


In [3]:
df_raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7043 non-null   str    
 1   SeniorCitizen     7043 non-null   int64  
 2   Partner           7043 non-null   str    
 3   Dependents        7043 non-null   str    
 4   tenure            7043 non-null   int64  
 5   PhoneService      7043 non-null   str    
 6   MultipleLines     7043 non-null   str    
 7   InternetService   7043 non-null   str    
 8   OnlineSecurity    7043 non-null   str    
 9   OnlineBackup      7043 non-null   str    
 10  DeviceProtection  7043 non-null   str    
 11  TechSupport       7043 non-null   str    
 12  StreamingTV       7043 non-null   str    
 13  StreamingMovies   7043 non-null   str    
 14  Contract          7043 non-null   str    
 15  PaperlessBilling  7043 non-null   str    
 16  PaymentMethod     7043 non-null   str    
 17  Monthl

In [4]:
# =====================
# 2. EDA and cleaning the raw data
# =====================

df_clean = df_raw.copy()

# Fix TotalCharges values to numeric
df_clean['TotalCharges'] = pd.to_numeric(df_clean['TotalCharges'], errors='coerce') # coerce - to convert non numeric values into NaN

# Check missing values
print('Gaps:')
print(df_clean.isna().sum())

# Check missing values after conversion
print('\nDoublecheck if there are any NaN created in TotalCharges:')
print(df_clean.isna().sum())

# Drop rows with missing values
df_clean = df_clean.dropna().reset_index(drop=True)

# Remove duplicates
initial_shape = df_clean.shape
df_clean = df_clean.drop_duplicates().reset_index(drop=True)
final_shape = df_clean.shape

# Print categorical value counts
print('\nCategorical feature counts:')
cat_cols = df_clean.select_dtypes(include=['object', 'str']).columns
for col in cat_cols:
    print(f'\n{col}:')
    print(df_clean[col].value_counts())

# Map target variable
df_clean['Churn'] = df_clean['Churn'].map({'Yes': 1, 'No': 0})

print(f'\nDuplicates removed: {initial_shape[0] - final_shape[0]}')
print(f'Final shape: {final_shape}')


Gaps:
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dtype: int64

Doublecheck if there are any NaN created in TotalCharges:
gender               0
SeniorCitizen        0
Partner              0
Dependents           0
tenure               0
PhoneService         0
MultipleLines        0
InternetService      0
OnlineSecurity       0
OnlineBackup         0
DeviceProtection     0
TechSupport          0
StreamingTV          0
StreamingMovies      0
Contract             0
PaperlessBilling     0
PaymentMethod        0
MonthlyCharges       0
TotalCharges        11
Churn                0
dt

In [5]:
print('\nCleaned dataset info:')
df_clean.info()
print('\nBasic statistics:')
display(df_clean.describe())


Cleaned dataset info:
<class 'pandas.DataFrame'>
RangeIndex: 7010 entries, 0 to 7009
Data columns (total 20 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   gender            7010 non-null   str    
 1   SeniorCitizen     7010 non-null   int64  
 2   Partner           7010 non-null   str    
 3   Dependents        7010 non-null   str    
 4   tenure            7010 non-null   int64  
 5   PhoneService      7010 non-null   str    
 6   MultipleLines     7010 non-null   str    
 7   InternetService   7010 non-null   str    
 8   OnlineSecurity    7010 non-null   str    
 9   OnlineBackup      7010 non-null   str    
 10  DeviceProtection  7010 non-null   str    
 11  TechSupport       7010 non-null   str    
 12  StreamingTV       7010 non-null   str    
 13  StreamingMovies   7010 non-null   str    
 14  Contract          7010 non-null   str    
 15  PaperlessBilling  7010 non-null   str    
 16  PaymentMethod     7010 non-nul

,SeniorCitizen,tenure,MonthlyCharges,TotalCharges,Churn
count,7010.000000,7010.000000,7010.000000,7010.000000,7010.000000
mean,0.162767,32.520399,64.888666,2290.353388,0.264907
std,0.369180,24.520441,30.064769,2266.820832,0.441315
min,0.000000,1.000000,18.250000,18.800000,0.000000
25%,0.000000,9.000000,35.750000,408.312500,0.000000
50%,0.000000,29.000000,70.400000,1403.875000,0.000000
75%,0.000000,56.000000,89.900000,3807.837500,1.000000
max,1.000000,72.000000,118.750000,8684.800000,1.000000


In [6]:
# =====================
# 3. Creating the log_model_results function
# =====================

from datetime import datetime

def log_model_results(model_name, model_params, search_time, scaler, metrics_train, metrics_test):
    file_path = os.path.join('csv_output', 'model_results.csv')
    ts = datetime.now().strftime('%Y-%m-%d %H:%M:%S')

    row_train = {
        'timestamp': ts,
        'model_name': model_name,
        'model_params': str(model_params),
        'search_time': search_time,
        'scaler': scaler,
        'recall': metrics_train.get('recall'),
        'precision': metrics_train.get('precision'),
        'f1': metrics_train.get('f1'),
        'roc_auc': metrics_train.get('roc_auc'),
        'best_cv_score': metrics_train.get('best_cv_score')
    }

    row_test = {
        'timestamp': ts,
        'model_name': f'{model_name}_TEST',
        'model_params': str(model_params),
        'search_time': search_time,
        'scaler': scaler,
        'recall': metrics_test.get('recall'),
        'precision': metrics_test.get('precision'),
        'f1': metrics_test.get('f1'),
        'roc_auc': metrics_test.get('roc_auc'),
        'best_cv_score': metrics_test.get('best_cv_score')
    }

    df_new = pd.DataFrame([row_train, row_test])

    # Append to csv without overwriting
    if os.path.exists(file_path):
        df_new.to_csv(file_path, mode='a', header=False, index=False)
    else:
        df_new.to_csv(file_path, mode='w', header=True, index=False)

    print('Results logged.')

In [8]:
# =====================
# 4. EDA Plots and Statistics
# =====================

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import math

print('Building EDA plots')

# Color map
color_map = {0: 'cyan', 1: 'pink'}

# Num and cat columns
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']
cat_cols = [c for c in df_clean.columns if c not in num_cols and c != 'Churn']

# 1. Target plot and Correlation Matrix
corr_matrix = df_clean[num_cols + ['Churn']].corr()

fig_tc = make_subplots(rows=1, cols=2, subplot_titles=('Target Distribution (Churn)', 'Correlation Matrix'),
                       specs=[[{"type": "xy"}, {"type": "heatmap"}]])

fig_tc.add_trace(go.Histogram(x=df_clean[df_clean['Churn']==0]['Churn'], name='No Churn', marker_color='cyan', showlegend=True), row=1, col=1)
fig_tc.add_trace(go.Histogram(x=df_clean[df_clean['Churn']==1]['Churn'], name='Churn', marker_color='pink', showlegend=True), row=1, col=1)

fig_tc.add_trace(go.Heatmap(z=corr_matrix.values, x=corr_matrix.columns, y=corr_matrix.index,
                            colorscale='Tealrose', zmin=-1, zmax=1, showscale=False, name='Correlation'), row=1, col=2)

fig_tc.update_layout(barmode='group', bargap=0.3, height=400, title_text='Target and Correlation')
fig_tc.show()

# 2. Categorical features
n_cols = 2
n_rows = math.ceil(len(cat_cols) / n_cols)
fig_cat = make_subplots(rows=n_rows, cols=n_cols, subplot_titles=cat_cols)

for i, col in enumerate(cat_cols):
    r = (i // n_cols) + 1
    c = (i % n_cols) + 1
    fig_cat.add_trace(go.Histogram(x=df_clean[df_clean['Churn']==0][col], name='No Churn', marker_color='cyan', showlegend=(i==0), legendgroup='No Churn'), row=r, col=c)
    fig_cat.add_trace(go.Histogram(x=df_clean[df_clean['Churn']==1][col], name='Churn', marker_color='pink', showlegend=(i==0), legendgroup='Churn'), row=r, col=c)

fig_cat.update_layout(barmode='group', bargap=0.3, bargroupgap=0.1, height=300*n_rows, title_text='Categorical Features')
fig_cat.show()

# 3. Numerical features
n_rows_num = math.ceil(len(num_cols) / n_cols)
fig_num = make_subplots(rows=n_rows_num, cols=n_cols, subplot_titles=num_cols)

for i, col in enumerate(num_cols):
    r = (i // n_cols) + 1
    c = (i % n_cols) + 1
    fig_num.add_trace(go.Histogram(x=df_clean[df_clean['Churn']==0][col], name='No Churn', marker_color='cyan', showlegend=(i==0), legendgroup='No Churn', opacity=0.6), row=r, col=c)
    fig_num.add_trace(go.Histogram(x=df_clean[df_clean['Churn']==1][col], name='Churn', marker_color='pink', showlegend=(i==0), legendgroup='Churn', opacity=0.6), row=r, col=c)

fig_num.update_layout(barmode='overlay', bargap=0.1, height=300*n_rows_num, title_text='Numerical Features')
fig_num.show()

Building EDA plots


In [11]:
# =====================
# 5. Data Preparation
# =====================

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# Create df_expanded
df_expanded = df_clean.copy()

# Define X and y
X = df_expanded.drop('Churn', axis=1)
y = df_expanded['Churn']

# Train/test split (80/20, stratified)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

print('Train shape:', X_train.shape)
print('Test shape:', X_test.shape)

# Identify categorical and numerical columns
cat_cols = X.select_dtypes(include=['object', 'str']).columns.tolist()
num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Create preprocessor (OHE for categorical, Scaler for numerical)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
    ]
)

# Combine preprocessor into a general pipeline placeholder
pipeline = Pipeline(steps=[('preprocessor', preprocessor)])


Train shape: (5608, 19)
Test shape: (1402, 19)


In [14]:
# =====================
# 6. Pipeline Setup
# =====================

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score

# Stratified K-Fold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def metrics_logger(model, model_name, search_time, scaler):
    # Predictions
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Probabilities for ROC-AUC
    y_train_proba = model.predict_proba(X_train)[:, 1]
    y_test_proba = model.predict_proba(X_test)[:, 1]

    # Train metrics (rounded to 4 decimal places)
    metrics_train = {
        'recall': round(recall_score(y_train, y_train_pred, zero_division=0), 4),
        'precision': round(precision_score(y_train, y_train_pred, zero_division=0), 4),
        'f1': round(f1_score(y_train, y_train_pred, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_train, y_train_proba), 4),
        'best_cv_score': round(model.best_score_, 4)
    }

    # Test metrics (rounded to 4 decimal places)
    metrics_test = {
        'recall': round(recall_score(y_test, y_test_pred, zero_division=0), 4),
        'precision': round(precision_score(y_test, y_test_pred, zero_division=0), 4),
        'f1': round(f1_score(y_test, y_test_pred, zero_division=0), 4),
        'roc_auc': round(roc_auc_score(y_test, y_test_proba), 4),
        'best_cv_score': round(model.best_score_, 4)
    }

    # Log results
    log_model_results(
        model_name=model_name,
        model_params=model.best_params_,
        search_time=round(search_time, 2),
        scaler=scaler,
        metrics_train=metrics_train,
        metrics_test=metrics_test
    )

    print(f'{model_name} evaluated and logged.')

In [25]:
# =====================
# 7. Model 1 - sklearn GradientBoosting
# =====================

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import RandomizedSearchCV
import time

# Create pipeline with GradientBoosting
pipeline_gb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', GradientBoostingClassifier(random_state=RANDOM_STATE))
])

# Parameter grid
param_grid_gb = {
    'model__n_estimators': [50, 100, 200],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__max_depth': [3, 5, 7]
}

# Randomized Search
search_gb = RandomizedSearchCV(
    estimator=pipeline_gb,
    param_distributions=param_grid_gb,
    n_iter=27,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

print('Training GradientBoosting')

# Time and fit
start_time = time.time()
search_gb.fit(X_train, y_train)
search_time = time.time() - start_time

# Log results
metrics_logger(
    model=search_gb,
    model_name='GradientBoosting',
    search_time=search_time,
    scaler='StandardScaler'
)

print('GradientBoosting finished')

Training GradientBoosting
Results logged.
GradientBoosting evaluated and logged.
GradientBoosting finished


In [24]:
# =====================
# 8. Model 2 - XGBoost
# =====================

from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV
import time

# Create pipeline with XGBoost
pipeline_xgb = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', XGBClassifier(random_state=RANDOM_STATE, eval_metric='logloss'))
])

# Parameter grid
param_grid_xgb = {
    'model__n_estimators': [50, 100, 200],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__max_depth': [3, 5, 7],
    'model__subsample': [0.8, 1.0]
}

# Randomized Search
search_xgb = RandomizedSearchCV(
    estimator=pipeline_xgb,
    param_distributions=param_grid_xgb,
    n_iter=50,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

print('Training XGBoost')

# Time and fit
start_time = time.time()
search_xgb.fit(X_train, y_train)
search_time = time.time() - start_time

# Log results
metrics_logger(
    model=search_xgb,
    model_name='XGBoost',
    search_time=search_time,
    scaler='StandardScaler'
)

print('XGBoost finished')

Training XGBoost
Results logged.
XGBoost evaluated and logged.
XGBoost finished


In [26]:
# =====================
# 9. Model 3 - LightGBM
# =====================

from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV
import time

# Create pipeline with LightGBM
pipeline_lgbm = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', LGBMClassifier(random_state=RANDOM_STATE))
])

# Parameter grid
param_grid_lgbm = {
    'model__n_estimators': [50, 100, 200],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__max_depth': [3, 5, 7],
    'model__num_leaves': [31, 50, 100]
}

# Randomized Search
search_lgbm = RandomizedSearchCV(
    estimator=pipeline_lgbm,
    param_distributions=param_grid_lgbm,
    n_iter=50,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

print('Training LightGBM')

# Time and fit
start_time = time.time()
search_lgbm.fit(X_train, y_train)
search_time = time.time() - start_time

# Log results
metrics_logger(
    model=search_lgbm,
    model_name='LightGBM',
    search_time=search_time,
    scaler='StandardScaler'
)

print('LightGBM finished')

Training LightGBM
[LightGBM] [Info] Number of positive: 1486, number of negative: 4122
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000376 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 668
[LightGBM] [Info] Number of data points in the train set: 5608, number of used features: 45
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.264979 -> initscore=-1.020251
[LightGBM] [Info] Start training from score -1.020251
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive 

In [27]:
# =====================
# 10. Model 4 - CatBoost
# =====================

from catboost import CatBoostClassifier
from sklearn.model_selection import RandomizedSearchCV
import time

# Create pipeline with CatBoost
# allow_writing_files=False prevents the 'catboost_info' directory creation error
pipeline_cat = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('model', CatBoostClassifier(random_state=RANDOM_STATE, verbose=0, allow_writing_files=False))
])

# Parameter grid
param_grid_cat = {
    'model__iterations': [50, 100, 200],
    'model__learning_rate': [0.01, 0.1, 0.2],
    'model__depth': [3, 5, 7]
}

# Randomized Search
search_cat = RandomizedSearchCV(
    estimator=pipeline_cat,
    param_distributions=param_grid_cat,
    n_iter=27,
    cv=skf,
    scoring='roc_auc',
    n_jobs=-1,
    random_state=RANDOM_STATE
)

print('Training CatBoost')

# Time and fit
start_time = time.time()
search_cat.fit(X_train, y_train)
search_time = time.time() - start_time

# Log results
metrics_logger(
    model=search_cat,
    model_name='CatBoost',
    search_time=search_time,
    scaler='StandardScaler'
)

print('CatBoost finished')

Training CatBoost
Results logged.
CatBoost evaluated and logged.
CatBoost finished


In [2]:
# =====================
# 11. Model Comparison
# =====================

import pandas as pd
import plotly.express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go

# Load logged results
results_path = 'csv_output/model_results.csv'
df_results = pd.read_csv(results_path)

# Extract base model name to find the latest run for each model
df_results['base_model'] = df_results['model_name'].str.replace('_TEST', '', regex=False)

# Find the latest timestamp for each base model
latest_timestamps = df_results.groupby('base_model')['timestamp'].max().reset_index()
latest_timestamps.columns = ['base_model', 'latest_timestamp']

# Filter dataframe to keep only rows matching the latest timestamps
df_latest = pd.merge(df_results, latest_timestamps, on='base_model')
df_latest = df_latest[df_latest['timestamp'] == df_latest['latest_timestamp']]

# Sort by base model and if it has _TEST suffix to keep train and test together
df_latest['is_test'] = df_latest['model_name'].str.contains('_TEST')
df_latest = df_latest.sort_values(by=['base_model', 'is_test']).reset_index(drop=True)

# Split into train and test dataframes
df_train = df_latest[~df_latest['is_test']]
df_test = df_latest[df_latest['is_test']]

# Plot comparison metrics (1 plot per metric)
metrics_to_plot = ['recall', 'precision', 'f1', 'roc_auc']
fig = make_subplots(rows=2, cols=2, subplot_titles=[m.upper() for m in metrics_to_plot])

for i, metric in enumerate(metrics_to_plot):
    row = (i // 2) + 1
    col = (i % 2) + 1

    # Bar for Train
    fig.add_trace(
        go.Bar(
            x=df_train['base_model'],
            y=df_train[metric],
            name='Train',
            marker_color='cyan',
            opacity=0.7,
            showlegend=(i == 0),
            width=0.35,
            offsetgroup=0
        ),
        row=row, col=col
    )

    # Bars for Test
    fig.add_trace(
        go.Bar(
            x=df_test['base_model'],
            y=df_test[metric],
            name='Test',
            marker_color='pink',
            opacity=0.7,
            showlegend=(i == 0),
            width=0.35,
            offsetgroup=1
        ),
        row=row, col=col
    )

# Set barmode to overlay so they stack on top of each other
fig.update_layout(barmode='group', height=700, title_text='Latest Model Runs Comparison (Cyan=Train, Pink=Test)', bargap=0.15, bargroupgap=0.1)
fig.show()

# Tabular view with compared values
# Create a temporary column to sort by the model's TEST recall
test_recall_map = df_test.set_index('base_model')['recall'].to_dict()
df_latest['sort_test_recall'] = df_latest['base_model'].map(test_recall_map)

# Sort by highest test recall first, then Train row before Test row
df_latest = df_latest.sort_values(by=['sort_test_recall', 'is_test'], ascending=[False, True]).reset_index(drop=True)

# Print sorted table
print('\nTabular View of Latest Runs (Sorted by Test Recall):')
display(df_latest[['timestamp', 'model_name', 'recall', 'precision', 'f1', 'roc_auc', 'best_cv_score', 'model_params']])

# Summary: find the best test run based on F1 score
best_row = df_test.loc[df_test['f1'].idxmax()]
best_name = best_row['base_model']
best_f1 = best_row['f1']
best_recall = best_row['recall']
best_precision = best_row['precision']
best_roc_auc = best_row['roc_auc']



Tabular View of Latest Runs (Sorted by Test Recall):


,timestamp,model_name,recall,precision,f1,roc_auc,best_cv_score,model_params
0,2026-08-16 18:01:24,CatBoost,0.5390,0.6899,0.6052,0.8614,0.8472,"{'model__learning_rate': 0.1, 'model__iteratio..."
1,2026-08-16 18:01:24,CatBoost_TEST,0.5202,0.6796,0.5893,0.8529,0.8472,"{'model__learning_rate': 0.1, 'model__iteratio..."
2,2026-08-16 17:58:56,XGBoost,0.5336,0.6914,0.6024,0.8646,0.8461,"{'model__subsample': 0.8, 'model__n_estimators..."
3,2026-08-16 17:58:56,XGBoost_TEST,0.5175,0.6857,0.5899,0.8507,0.8461,"{'model__subsample': 0.8, 'model__n_estimators..."
4,2026-08-16 18:00:58,LightGBM,0.5363,0.6936,0.6049,0.8652,0.8453,"{'model__num_leaves': 50, 'model__n_estimators..."
5,2026-08-16 18:00:58,LightGBM_TEST,0.5148,0.6773,0.5850,0.8504,0.8453,"{'model__num_leaves': 50, 'model__n_estimators..."
6,2026-08-16 18:00:20,GradientBoosting,0.5363,0.7034,0.6086,0.8681,0.8448,"{'model__n_estimators': 50, 'model__max_depth'..."
7,2026-08-16 18:00:20,GradientBoosting_TEST,0.5094,0.6799,0.5824,0.8513,0.8448,"{'model__n_estimators': 50, 'model__max_depth'..."
